# 04 — Model Registry with Aliases

**What you'll learn:**
- Register a model in the MLflow Model Registry
- Use **aliases** (`@champion`, `@challenger`) for lifecycle management —
  the modern replacement for the deprecated `Staging`/`Production` stages
- Load a registered model by alias and predict

**Prerequisite:** Run template 01, 02, or 03 first to have a logged model.
Set `SOURCE_RUN_ID` below to a run ID that has a logged model artifact.

**Note:** MLflow 2.9+ deprecated model stages in favour of aliases and tags.
Aliases are mutable named pointers to a specific version (e.g. `@champion`
always resolves to whatever version is currently blessed).

## Setup

In [ ]:
import mlflow
from mlflow import MlflowClient
import numpy as np

# Make sibling config.py importable (VS Code or Jupyter, any cwd)
import sys
from pathlib import Path
try:
    _HERE = Path(__vsc_ipynb_file__).parent
except NameError:
    _HERE = Path.cwd()
sys.path.insert(0, str(_HERE))

from config import init_mlflow

EXPERIMENT_NAME = "04-registry-demo"
init_mlflow(experiment_name=EXPERIMENT_NAME)

# ── Set this to a run ID from a previous template ────────────────────────
# MLflow 3: paste the MODEL_URI printed by template 01/02 (e.g. "models:/m-abc…")
# OR the run_id — we'll resolve it via the "model_uri" tag set by template 01.
SOURCE_RUN_ID = "938fe1905f074a70ba194d1b46868106"
REGISTERED_MODEL_NAME = "churn-classifier"

client = MlflowClient()

/home/rami/Work/kyper/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/15 21:28:50 INFO mlflow.tracking.fluent: Experiment with name '04-registry-demo' does not exist. Creating a new experiment.


✅ MLflow tracking URI: file:///home/rami/Work/kyper/kyper-framework/notebooks/tutorial/mlruns
✅ MLflow experiment:   04-registry-demo


## Step 1 — Register the model

This creates (or updates) a named model in the MLflow registry,
versioned automatically.

In [ ]:
# Resolve the model URI from the source run.
# - Preferred: the `model_uri` tag set by template 01 at log time
#   (`mlflow.set_tag("model_uri", model_info.model_uri)`).
# - Fallback: the canonical `runs:/<run_id>/<artifact_path>` form, which
#   works for any run that logged a model under `name="model"` (e.g.
#   template 02's final-best-model cell).
source_run = client.get_run(SOURCE_RUN_ID)
model_uri = source_run.data.tags.get("model_uri") or f"runs:/{SOURCE_RUN_ID}/model"
print(f"Registering from: {model_uri}")

registered = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME,
)

version = registered.version
print(f"✅ Registered '{REGISTERED_MODEL_NAME}' — version {version}")

## Step 2 — Add description and tags

In [ ]:
client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=version,
    description="Random Forest trained on synthetic churn data. "
                "Optimised for F1 score.",
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=version,
    key="dataset_version",
    value="synthetic-v1",
)
client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=version,
    key="framework",
    value="scikit-learn",
)

print(f"✅ Tags and description set for version {version}")

## Step 3 — Assign the `@challenger` alias

Aliases are mutable pointers. Use `@challenger` for a version under
evaluation and `@champion` for the live production version.

In [ ]:
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="challenger",
    version=version,
)
print(f"✅ {REGISTERED_MODEL_NAME}@challenger → version {version}")

## Step 4 — Validate the challenger

Load the aliased model and validate on a holdout set before promotion.

In [ ]:
challenger = mlflow.sklearn.load_model(
    f"models:/{REGISTERED_MODEL_NAME}@challenger"
)

# Quick sanity check — replace with your real validation data
X_holdout = np.random.randn(100, 20)
preds = challenger.predict(X_holdout)
assert preds.shape == (100,), "Prediction shape mismatch"
print(f"✅ Challenger validation passed. Sample predictions: {preds[:5]}")

with mlflow.start_run(run_name="challenger-validation"):
    mlflow.set_tags({
        "stage": "challenger-validation",
        "model_name": REGISTERED_MODEL_NAME,
        "model_version": version,
    })
    mlflow.log_metric("n_validation_samples", 100)
    mlflow.log_metric("validation_passed", 1)

## Step 5 — Promote challenger to champion

Reassigning the alias is atomic — no need to archive the previous version.

In [ ]:
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="champion",
    version=version,
)

# Remove the challenger alias now that it's the champion
client.delete_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="challenger",
)

print(f"✅ {REGISTERED_MODEL_NAME}@champion → version {version}")

## Step 6 — Load the champion and predict

In [ ]:
champion = mlflow.sklearn.load_model(
    f"models:/{REGISTERED_MODEL_NAME}@champion"
)

X_sample = np.random.randn(5, 20)
predictions = champion.predict(X_sample)
print(f"Champion predictions: {predictions}")

## Step 7 — Inspect aliases and versions

In [ ]:
mv = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "champion")
print(f"Champion is version {mv.version}")
print(f"  run_id: {mv.run_id}")
print(f"  tags:   {mv.tags}")

# All versions
versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
for v in versions:
    print(f"  v{v.version}  aliases={v.aliases}  tags={dict(v.tags)}")

## Alias lifecycle

```
mlflow.register_model()
        │
        ▼
    version N  ◄── set_registered_model_alias("challenger", N)
        │              │
        │   validate   │
        ▼              ▼
    set_registered_model_alias("champion", N)
    delete_registered_model_alias("challenger")
```

For serving, see template `06_mlflow_model_serve.py`.